In [ ]:
!pip install shap
!pip install openpyxl


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import shap
import warnings
warnings.filterwarnings("ignore")

sns.set(style='whitegrid')


In [ ]:
#  Load Dataset and Preprocess
df = pd.read_excel("LP_data_with_dates.xlsx", sheet_name="DATA")

# Drop timestamp (not useful for modeling)
df.drop(columns=["TIMESTAMP"], inplace=True)

# Handle missing values
df.fillna(df.mean(), inplace=True)

# Quick view
df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'LP_data_with_dates.xlsx'

In [ ]:
#Plot TOTAL_FLOW distribution
plt.figure(figsize=(10, 6))
sns.histplot(df["TOTAL_FLOW"], kde=True, bins=50, color="steelblue")
plt.title("Distribution of TOTAL_FLOW", fontsize=14)
plt.xlabel("TOTAL_FLOW")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
#Train-Test Split (80-20)
X = df.drop(columns=["TOTAL_FLOW"])
y = df["TOTAL_FLOW"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
# Train Models and Evaluate
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state=42),
    "XGBoost": XGBRegressor(random_state=42, verbosity=0)
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    results[name] = {
        "RMSE": np.sqrt(mean_squared_error(y_test, preds)),
        "MAE": mean_absolute_error(y_test, preds),
        "R2": r2_score(y_test, preds),
        "model": model,
        "preds": preds
    }

pd.DataFrame({
    name: {
        "RMSE": r["RMSE"],
        "MAE": r["MAE"],
        "R²": r["R2"]
    }
    for name, r in results.items()
}).T


In [ ]:
#Prediction Plots (Actual vs Predicted)
plt.figure(figsize=(15, 5))

for i, (name, r) in enumerate(results.items(), 1):
    plt.subplot(1, 3, i)
    sns.scatterplot(x=y_test, y=r["preds"], alpha=0.3, edgecolor=None)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
    plt.xlabel("Actual TOTAL_FLOW")
    plt.ylabel("Predicted TOTAL_FLOW")
    plt.title(name)

plt.tight_layout()
plt.show()


In [ ]:
# SHAP summary plots for all three models
for name in ["Linear Regression", "Random Forest", "XGBoost"]:
    print(f"🔍 SHAP Interpretation for: {name}")

    model = results[name]["model"]

    # Use TreeExplainer for tree models, LinearExplainer for linear model
    if name == "Linear Regression":
        explainer = shap.Explainer(model, X_train)
    else:
        explainer = shap.Explainer(model, X_test)

    shap_values = explainer(X_test)

    # Show summary beeswarm plot
    shap.plots.beeswarm(shap_values, max_display=15)


In [ ]:
import joblib

# Save each model from the results dictionary
joblib.dump(results["Linear Regression"]["model"], "lin_model.pkl")
joblib.dump(results["Random Forest"]["model"], "rf_model.pkl")
joblib.dump(results["XGBoost"]["model"], "xgb_model.pkl")

print("✅ Models saved successfully!")
